# C5 — ESC-50 : référence MFCC + SVM (P3, P4, P5)

Ce notebook construit le manifeste d'ESC-50, résume chaque extrait en 240 nombres (MFCC, Δ, ΔΔ :
moyenne et écart-type dans le temps), puis évalue un SVM par validation croisée sur les
5 folds officiels. Tout le calcul vient du dépôt (`echolens.core`, `echolens.training`) :
le notebook ne fait qu'enchaîner les étapes et enregistrer les résultats dans `/kaggle/working`.

Référence publiée : Piczak (2015), MFCC + SVM, 39,6 % d'accuracy.

## a. Démarrage

On clone le dépôt (branche `BRANCH`) et on l'installe en mode éditable (I9). L'installation
éditable n'est visible qu'après un redémarrage du noyau : on ajoute donc aussi `src/` au
`sys.path` pour pouvoir importer `echolens` tout de suite.

In [ ]:
import sys

BRANCH = "main"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. ffmpeg

`echolens.core.audio.load_audio` décode tous les formats avec ffmpeg (D1). On vérifie qu'il est
installé, sinon on l'installe.

In [ ]:
import shutil

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
print("ffmpeg :", shutil.which("ffmpeg"))

## c. Trouver ESC-50

On cherche `meta/esc50.csv` dans les datasets attachés (`/kaggle/input`). S'il est absent, on
télécharge l'archive officielle depuis GitHub. Le jeu doit se trouver dans un dossier nommé
exactement `esc50` : `datasets_dir` est son dossier parent (contrat des manifestes).

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

ARCHIVE_URL = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"
LOCAL_DATASETS_DIR = Path("/kaggle/working/datasets")

found = sorted(Path("/kaggle/input").glob("**/meta/esc50.csv"))
if found:
    esc50_folder = Path(found[0]).parent.parent
    print("ESC-50 trouvé :", esc50_folder)
else:
    # Téléchargement (~600 Mo) puis extraction dans /kaggle/working/datasets/esc50.
    LOCAL_DATASETS_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = LOCAL_DATASETS_DIR / "esc50.zip"
    urllib.request.urlretrieve(ARCHIVE_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(LOCAL_DATASETS_DIR)
    archive_path.unlink()
    (LOCAL_DATASETS_DIR / "ESC-50-master").rename(LOCAL_DATASETS_DIR / "esc50")
    esc50_folder = LOCAL_DATASETS_DIR / "esc50"
    print("ESC-50 téléchargé :", esc50_folder)

# Le dossier doit s'appeler « esc50 » : sinon, un lien symbolique portant ce nom.
if esc50_folder.name != "esc50":
    LOCAL_DATASETS_DIR.mkdir(parents=True, exist_ok=True)
    link = LOCAL_DATASETS_DIR / "esc50"
    if not link.exists():
        link.symlink_to(esc50_folder)
    esc50_folder = link

datasets_dir = esc50_folder.parent
print("datasets_dir =", datasets_dir)

## d. Manifeste

`build_esc50_manifest` lit `meta/esc50.csv` (fold, classe, enregistrement d'origine) et vérifie
qu'aucune source n'est partagée entre deux folds (P3). On attend 2 000 lignes, 400 par fold.

In [ ]:
from collections import Counter

from echolens.core.data.esc50 import build_esc50_manifest
from echolens.core.data.manifest import dataset_root, save_manifest

rows = build_esc50_manifest(dataset_root("esc50", datasets_dir))
save_manifest(rows, Path("/kaggle/working/esc50.csv"))

rows_per_fold = Counter(row.fold for row in rows)
print("Lignes :", len(rows))
for fold in sorted(rows_per_fold):
    print("fold", fold, ":", rows_per_fold[fold], "lignes")

## e. Vérifier les 5 catégories

La matrice agrégée suppose que `target // 10` donne les 5 grandes catégories d'ESC-50. On le
vérifie sur le vrai CSV : la première classe de chaque groupe de 10 doit être dog, rain,
crying_baby, door_wood_knock et helicopter. Sinon, on arrête le notebook.

In [ ]:
import csv

EXPECTED_FIRST_CLASSES = ["dog", "rain", "crying_baby", "door_wood_knock", "helicopter"]

category_of_target = {}
metadata_path = esc50_folder / "meta" / "esc50.csv"
with metadata_path.open(encoding="utf-8") as handle:
    for record in csv.DictReader(handle):
        category_of_target[int(record["target"])] = record["category"]

class_names = [category_of_target[target] for target in range(50)]
first_classes = [class_names[group * 10] for group in range(5)]
print("Premières classes de chaque groupe :", first_classes)
if first_classes != EXPECTED_FIRST_CLASSES:
    raise RuntimeError("target // 10 ne donne pas les 5 catégories attendues : arrêt.")

## f. Features et validation croisée

Chaque extrait est chargé en mono 16 kHz par `load_audio`, puis résumé en 240 nombres. Le
résultat est mis en cache dans `/kaggle/working` : une relance ne recalcule rien. Ensuite,
5 tours : pour le fold de test k, C est choisi sur le fold k + 1, puis le SVM est réentraîné
sur les 4 folds hors test et évalué une seule fois sur le fold k.

In [ ]:
import time

import pandas as pd

from echolens.config import CONFIG_DIR
from echolens.core.features import FeatureConfig
from echolens.training.esc50_svm import SvmConfig, cross_validate, extract_features

feature_config = FeatureConfig.from_yaml(CONFIG_DIR / "features.yaml")
svm_config = SvmConfig.from_yaml(CONFIG_DIR / "baseline_svm.yaml")
cache_path = Path("/kaggle/working/esc50_mfcc_features.npz")

start = time.time()
features, targets, folds = extract_features(rows, datasets_dir, feature_config, cache_path)
print("Features :", features.shape, f"en {time.time() - start:.0f} s")

start = time.time()
result = cross_validate(features, targets, folds, svm_config)
print(f"Validation croisée en {time.time() - start:.0f} s")

fold_rows = []
for fold_result in result.folds:
    fold_rows.append(
        {
            "fold de test": fold_result.test_fold,
            "C retenu": fold_result.chosen_c,
            "F1 macro": round(fold_result.macro_f1, 4),
            "accuracy": round(fold_result.accuracy, 4),
        }
    )
display(pd.DataFrame(fold_rows))
print(f"F1 macro : {result.macro_f1_mean:.4f} ± {result.macro_f1_std:.4f}")
print(f"Accuracy : {result.accuracy_mean:.4f} ± {result.accuracy_std:.4f}")

## g. Résultats en JSON

Moyennes ± écarts-types, résultats par fold, C retenus, date, version exacte du dépôt et
paramètres utilisés : de quoi reproduire et citer le résultat dans le rapport.

In [ ]:
import datetime
import json
import subprocess

import yaml

commit = subprocess.run(
    ["git", "-C", "/kaggle/working/TP_SIGNAL_DS", "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

per_fold = []
for fold_result in result.folds:
    per_fold.append(
        {
            "test_fold": fold_result.test_fold,
            "chosen_c": fold_result.chosen_c,
            "macro_f1": fold_result.macro_f1,
            "accuracy": fold_result.accuracy,
        }
    )

report = {
    "experiment": "esc50_mfcc_svm",
    "date": datetime.datetime.now(datetime.UTC).isoformat(),
    "branch": BRANCH,
    "commit": commit,
    "macro_f1_mean": result.macro_f1_mean,
    "macro_f1_std": result.macro_f1_std,
    "accuracy_mean": result.accuracy_mean,
    "accuracy_std": result.accuracy_std,
    "chosen_c": [fold_result.chosen_c for fold_result in result.folds],
    "per_fold": per_fold,
    "features_config": yaml.safe_load((CONFIG_DIR / "features.yaml").read_text(encoding="utf-8")),
    "svm_config": yaml.safe_load((CONFIG_DIR / "baseline_svm.yaml").read_text(encoding="utf-8")),
}
json_path = Path("/kaggle/working/esc50_mfcc_svm.json")
with json_path.open("w", encoding="utf-8") as handle:
    json.dump(report, handle, ensure_ascii=False, indent=2)
print(json.dumps(report, ensure_ascii=False, indent=2)[:1500])

## h. Matrice de confusion en 5 catégories et classes les plus confondues

Matrice normalisée par ligne et cumulée sur les 5 folds de test : la case (i, j) est la part
des extraits de la catégorie i prédits dans la catégorie j. Puis les 5 paires (vraie classe →
classe prédite) les plus fréquentes hors diagonale, sur la matrice en 50 classes.

In [ ]:
import matplotlib.pyplot as plt

category_names = svm_config.category_names
matrix = result.confusion_categories

figure, axes = plt.subplots(figsize=(8, 6))
image = axes.imshow(matrix, cmap="Blues", vmin=0.0, vmax=1.0)
axes.set_xticks(range(len(category_names)), category_names, rotation=30, ha="right")
axes.set_yticks(range(len(category_names)), category_names)
axes.set_xlabel("Catégorie prédite")
axes.set_ylabel("Vraie catégorie")
axes.set_title("ESC-50, MFCC + SVM (5 folds)")
for i in range(len(category_names)):
    for j in range(len(category_names)):
        color = "white" if matrix[i, j] > 0.5 else "black"
        axes.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", color=color)
figure.colorbar(image, ax=axes, label="Part de la vraie catégorie")
figure.tight_layout()
figure.savefig("/kaggle/working/esc50_svm_confusion_categories.png", dpi=150)
plt.show()

pairs = []
for true_class in range(50):
    for predicted_class in range(50):
        if true_class != predicted_class:
            rate = result.confusion_classes[true_class, predicted_class]
            pairs.append((rate, class_names[true_class], class_names[predicted_class]))
pairs.sort(reverse=True)
print("Paires les plus confondues (vraie → prédite, part des extraits de la vraie classe) :")
for rate, true_name, predicted_name in pairs[:5]:
    print(f"  {true_name} → {predicted_name} : {rate:.1%}")

## Nettoyage

On supprime le jeu téléchargé et le clone du dépôt : les sorties du notebook ne gardent que
le manifeste, le cache des features, le JSON et la figure.

In [ ]:
shutil.rmtree(LOCAL_DATASETS_DIR, ignore_errors=True)
shutil.rmtree(CLONE_DIR, ignore_errors=True)
remaining = [path.name for path in Path("/kaggle/working").iterdir()]
print(sorted(remaining))